# 服飾資料庫連結應用 — Filess.io MySQL 連線與 SQL 完整驗證 (A1 ~ C2)
學號：13460190 ｜ 姓名：蔡曜安

In [3]:
!pip install pymysql sqlalchemy pandas -q

import json
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# 請填入 Filess.io 面板上的 5 項連線參數（注意 Port 為面板顯示的 5 位數）
DB_HOST = "7rieeo.h.filess.io"       # 例如: xy123.h.filess.io
DB_PORT = 3307                  # 請改成 Filess.io 面板顯示的實際 Port
DB_USER = "uuuuuuuu_boxrealize"
DB_PASS = "99c53564b6b4ed7967e0199ccc1fc9749e277159"
DB_NAME = "uuuuuuuu_boxrealize"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}?charset=utf8mb4"
)
print("Filess.io MySQL 資料庫引擎建立成功！")

Filess.io MySQL 資料庫引擎建立成功！


In [8]:
# 步驟 1：建立 6 張資料表並寫入全部 226 筆資料（含 image_url 與 60 張照片一對一對應）
raw_sql = """-- ====================================================================
-- 服飾資料庫連結應用 (Filess.io MySQL 專用完整建表、匯入與作業查詢腳本)
-- 學號：13460190 ｜ 姓名：蔡曜安
-- ====================================================================
SET FOREIGN_KEY_CHECKS = 0;
DROP TABLE IF EXISTS stock_reservation;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS clothing;
DROP TABLE IF EXISTS brands;
DROP TABLE IF EXISTS categories;
SET FOREIGN_KEY_CHECKS = 1;

-- 1. categories
CREATE TABLE categories (category_id INT PRIMARY KEY, category_name VARCHAR(50) NOT NULL);
INSERT INTO categories VALUES (1, '上衣');
INSERT INTO categories VALUES (2, '褲子');
INSERT INTO categories VALUES (3, '裙子');
INSERT INTO categories VALUES (4, '外套');
INSERT INTO categories VALUES (5, '洋裝');
INSERT INTO categories VALUES (6, '運動服');

-- 2. brands
CREATE TABLE brands (brand_id INT PRIMARY KEY, brand_name VARCHAR(100) NOT NULL, country VARCHAR(50) NOT NULL);
INSERT INTO brands VALUES (1, 'Urban Leaf', '台灣');
INSERT INTO brands VALUES (2, 'North Thread', '日本');
INSERT INTO brands VALUES (3, 'Mode Studio', '韓國');
INSERT INTO brands VALUES (4, 'Daily Wear', '台灣');
INSERT INTO brands VALUES (5, 'Peak Active', '美國');

-- 3. clothing (含 image_url 欄位與 60 張照片一對一類別精準對應)
CREATE TABLE clothing (
    product_id VARCHAR(10) PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    category_id INT NOT NULL,
    brand_id INT NOT NULL,
    color VARCHAR(30),
    size VARCHAR(10),
    price DECIMAL(10,2) NOT NULL,
    stock_qty INT NOT NULL,
    listed_date DATE NOT NULL,
    image_url VARCHAR(255) NOT NULL,
    FOREIGN KEY (category_id) REFERENCES categories(category_id),
    FOREIGN KEY (brand_id) REFERENCES brands(brand_id)
);
INSERT INTO clothing VALUES ('P0001', '純棉T恤 01款', 1, 1, '白色', 'M', 610, 7, '2026-01-01', 'images/products/product-021.jpg');
INSERT INTO clothing VALUES ('P0002', '直筒牛仔褲 02款', 2, 2, '米色', 'L', 830, 14, '2026-01-01', 'images/products/product-041.jpg');
INSERT INTO clothing VALUES ('P0003', '百褶裙 03款', 3, 3, '藍色', 'XL', 1050, 21, '2025-06-15', 'images/products/product-035.jpg');
INSERT INTO clothing VALUES ('P0004', '防風外套 04款', 4, 4, '灰色', 'S', 1270, 28, '2026-01-01', 'images/products/product-049.jpg');
INSERT INTO clothing VALUES ('P0005', '碎花洋裝 05款', 5, 5, '綠色', 'M', 1490, 35, '2026-01-01', 'images/products/product-001.jpg');
INSERT INTO clothing VALUES ('P0006', '運動上衣 06款', 6, 1, '黑色', 'L', 1710, 4, '2025-06-15', 'images/products/product-055.jpg');
INSERT INTO clothing VALUES ('P0007', '襯衫 07款', 1, 2, '白色', 'XL', 1930, 11, '2026-01-01', 'images/products/product-016.jpg');
INSERT INTO clothing VALUES ('P0008', '休閒長褲 08款', 2, 3, '米色', 'S', 2150, 18, '2026-01-01', 'images/products/product-045.jpg');
INSERT INTO clothing VALUES ('P0009', '牛仔短裙 09款', 3, 4, '藍色', 'M', 2370, 25, '2025-06-15', 'images/products/product-034.jpg');
INSERT INTO clothing VALUES ('P0010', '丹寧外套 10款', 4, 5, '灰色', 'L', 500, 32, '2026-01-01', 'images/products/product-051.jpg');
INSERT INTO clothing VALUES ('P0011', '無袖洋裝 11款', 5, 1, '綠色', 'XL', 720, 1, '2026-01-01', 'images/products/product-007.jpg');
INSERT INTO clothing VALUES ('P0012', '瑜伽褲 12款', 6, 2, '黑色', 'S', 940, 8, '2025-06-15', 'images/products/product-048.jpg');
INSERT INTO clothing VALUES ('P0013', '針織上衣 13款', 1, 3, '白色', 'M', 1160, 15, '2026-01-01', 'images/products/product-024.jpg');
INSERT INTO clothing VALUES ('P0014', '西裝褲 14款', 2, 4, '米色', 'L', 1380, 22, '2026-01-01', 'images/products/product-046.jpg');
INSERT INTO clothing VALUES ('P0015', '長裙 15款', 3, 5, '藍色', 'XL', 1600, 29, '2025-06-15', 'images/products/product-039.jpg');
INSERT INTO clothing VALUES ('P0016', '連帽外套 16款', 4, 1, '灰色', 'S', 1820, 36, '2026-01-01', 'images/products/product-052.jpg');
INSERT INTO clothing VALUES ('P0017', '長袖洋裝 17款', 5, 2, '綠色', 'M', 2040, 5, '2026-01-01', 'images/products/product-006.jpg');
INSERT INTO clothing VALUES ('P0018', '訓練短褲 18款', 6, 3, '黑色', 'L', 2260, 12, '2025-06-15', 'images/products/product-015.jpg');
INSERT INTO clothing VALUES ('P0019', '短袖上衣 19款', 1, 4, '白色', 'XL', 390, 19, '2026-01-01', 'images/products/product-014.jpg');
INSERT INTO clothing VALUES ('P0020', '寬褲 20款', 2, 5, '米色', 'S', 610, 26, '2026-01-01', 'images/products/product-042.jpg');
INSERT INTO clothing VALUES ('P0021', 'A字裙 21款', 3, 1, '藍色', 'M', 830, 33, '2025-06-15', 'images/products/product-038.jpg');
INSERT INTO clothing VALUES ('P0022', '西裝外套 22款', 4, 2, '灰色', 'L', 1050, 2, '2026-01-01', 'images/products/product-058.jpg');
INSERT INTO clothing VALUES ('P0023', '襯衫洋裝 23款', 5, 3, '綠色', 'XL', 1270, 9, '2026-01-01', 'images/products/product-003.jpg');
INSERT INTO clothing VALUES ('P0024', '運動外套 24款', 6, 4, '黑色', 'S', 1490, 16, '2025-06-15', 'images/products/product-056.jpg');
INSERT INTO clothing VALUES ('P0025', '條紋上衣 25款', 1, 5, '白色', 'M', 1710, 23, '2026-01-01', 'images/products/product-026.jpg');
INSERT INTO clothing VALUES ('P0026', '棉質短褲 26款', 2, 1, '米色', 'L', 1930, 30, '2026-01-01', 'images/products/product-032.jpg');
INSERT INTO clothing VALUES ('P0027', '針織裙 27款', 3, 2, '藍色', 'XL', 2150, 37, '2025-06-15', 'images/products/product-037.jpg');
INSERT INTO clothing VALUES ('P0028', '羊毛外套 28款', 4, 3, '灰色', 'S', 2370, 6, '2026-01-01', 'images/products/product-057.jpg');
INSERT INTO clothing VALUES ('P0029', '針織洋裝 29款', 5, 4, '綠色', 'M', 500, 13, '2026-01-01', 'images/products/product-009.jpg');
INSERT INTO clothing VALUES ('P0030', '慢跑長褲 30款', 6, 5, '黑色', 'L', 720, 20, '2025-06-15', 'images/products/product-020.jpg');
INSERT INTO clothing VALUES ('P0031', '純棉T恤 31款', 1, 1, '白色', 'XL', 940, 27, '2026-01-01', 'images/products/product-011.jpg');
INSERT INTO clothing VALUES ('P0032', '直筒牛仔褲 32款', 2, 2, '米色', 'S', 1160, 34, '2026-01-01', 'images/products/product-047.jpg');
INSERT INTO clothing VALUES ('P0033', '百褶裙 33款', 3, 3, '藍色', 'M', 1380, 3, '2025-06-15', 'images/products/product-036.jpg');
INSERT INTO clothing VALUES ('P0034', '防風外套 34款', 4, 4, '灰色', 'L', 1600, 10, '2026-01-01', 'images/products/product-050.jpg');
INSERT INTO clothing VALUES ('P0035', '碎花洋裝 35款', 5, 5, '綠色', 'XL', 1820, 17, '2026-01-01', 'images/products/product-002.jpg');
INSERT INTO clothing VALUES ('P0036', '運動上衣 36款', 6, 1, '黑色', 'S', 2040, 24, '2025-06-15', 'images/products/product-054.jpg');
INSERT INTO clothing VALUES ('P0037', '襯衫 37款', 1, 2, '白色', 'M', 2260, 31, '2026-01-01', 'images/products/product-017.jpg');
INSERT INTO clothing VALUES ('P0038', '休閒長褲 38款', 2, 3, '米色', 'L', 390, 0, '2026-01-01', 'images/products/product-043.jpg');
INSERT INTO clothing VALUES ('P0039', '牛仔短裙 39款', 3, 4, '藍色', 'XL', 610, 7, '2025-06-15', 'images/products/product-030.jpg');
INSERT INTO clothing VALUES ('P0040', '丹寧外套 40款', 4, 5, '灰色', 'S', 830, 14, '2026-01-01', 'images/products/product-053.jpg');
INSERT INTO clothing VALUES ('P0041', '無袖洋裝 41款', 5, 1, '綠色', 'M', 1050, 21, '2026-01-01', 'images/products/product-008.jpg');
INSERT INTO clothing VALUES ('P0042', '瑜伽褲 42款', 6, 2, '黑色', 'L', 1270, 28, '2025-06-15', 'images/products/product-022.jpg');
INSERT INTO clothing VALUES ('P0043', '針織上衣 43款', 1, 3, '白色', 'XL', 1490, 35, '2026-01-01', 'images/products/product-029.jpg');
INSERT INTO clothing VALUES ('P0044', '西裝褲 44款', 2, 4, '米色', 'S', 1710, 4, '2026-01-01', 'images/products/product-044.jpg');
INSERT INTO clothing VALUES ('P0045', '長裙 45款', 3, 5, '藍色', 'M', 1930, 11, '2025-06-15', 'images/products/product-040.jpg');
INSERT INTO clothing VALUES ('P0046', '連帽外套 46款', 4, 1, '灰色', 'L', 2150, 18, '2026-01-01', 'images/products/product-025.jpg');
INSERT INTO clothing VALUES ('P0047', '長袖洋裝 47款', 5, 2, '綠色', 'XL', 2370, 25, '2026-01-01', 'images/products/product-005.jpg');
INSERT INTO clothing VALUES ('P0048', '訓練短褲 48款', 6, 3, '黑色', 'S', 500, 32, '2025-06-15', 'images/products/product-013.jpg');
INSERT INTO clothing VALUES ('P0049', '短袖上衣 49款', 1, 4, '白色', 'M', 720, 1, '2026-01-01', 'images/products/product-012.jpg');
INSERT INTO clothing VALUES ('P0050', '寬褲 50款', 2, 5, '米色', 'L', 940, 8, '2026-01-01', 'images/products/product-033.jpg');
INSERT INTO clothing VALUES ('P0051', 'A字裙 51款', 3, 1, '藍色', 'XL', 1160, 15, '2025-06-15', 'images/products/product-027.jpg');
INSERT INTO clothing VALUES ('P0052', '西裝外套 52款', 4, 2, '灰色', 'S', 1380, 22, '2026-01-01', 'images/products/product-060.jpg');
INSERT INTO clothing VALUES ('P0053', '襯衫洋裝 53款', 5, 3, '綠色', 'M', 1600, 29, '2026-01-01', 'images/products/product-004.jpg');
INSERT INTO clothing VALUES ('P0054', '運動外套 54款', 6, 4, '黑色', 'L', 1820, 36, '2025-06-15', 'images/products/product-019.jpg');
INSERT INTO clothing VALUES ('P0055', '條紋上衣 55款', 1, 5, '白色', 'XL', 2040, 5, '2026-01-01', 'images/products/product-018.jpg');
INSERT INTO clothing VALUES ('P0056', '棉質短褲 56款', 2, 1, '米色', 'S', 2260, 12, '2026-01-01', 'images/products/product-031.jpg');
INSERT INTO clothing VALUES ('P0057', '針織裙 57款', 3, 2, '藍色', 'M', 390, 19, '2025-06-15', 'images/products/product-028.jpg');
INSERT INTO clothing VALUES ('P0058', '羊毛外套 58款', 4, 3, '灰色', 'L', 610, 26, '2026-01-01', 'images/products/product-059.jpg');
INSERT INTO clothing VALUES ('P0059', '針織洋裝 59款', 5, 4, '綠色', 'XL', 830, 33, '2026-01-01', 'images/products/product-010.jpg');
INSERT INTO clothing VALUES ('P0060', '慢跑長褲 60款', 6, 5, '黑色', 'S', 1050, 2, '2025-06-15', 'images/products/product-023.jpg');

-- 4. customers
CREATE TABLE customers (customer_id VARCHAR(10) PRIMARY KEY, customer_name VARCHAR(100) NOT NULL, city VARCHAR(50), member_level VARCHAR(20));
INSERT INTO customers VALUES ('C001', '顧客001', '台北', '銀卡');
INSERT INTO customers VALUES ('C002', '顧客002', '新竹', '金卡');
INSERT INTO customers VALUES ('C003', '顧客003', '台中', '一般');
INSERT INTO customers VALUES ('C004', '顧客004', '高雄', '銀卡');
INSERT INTO customers VALUES ('C005', '顧客005', '桃園', '金卡');
INSERT INTO customers VALUES ('C006', '顧客006', '台北', '一般');
INSERT INTO customers VALUES ('C007', '顧客007', '新竹', '銀卡');
INSERT INTO customers VALUES ('C008', '顧客008', '台中', '金卡');
INSERT INTO customers VALUES ('C009', '顧客009', '高雄', '一般');
INSERT INTO customers VALUES ('C010', '顧客010', '桃園', '銀卡');
INSERT INTO customers VALUES ('C011', '顧客011', '台北', '金卡');
INSERT INTO customers VALUES ('C012', '顧客012', '新竹', '一般');
INSERT INTO customers VALUES ('C013', '顧客013', '台中', '銀卡');
INSERT INTO customers VALUES ('C014', '顧客014', '高雄', '金卡');
INSERT INTO customers VALUES ('C015', '顧客015', '桃園', '一般');
INSERT INTO customers VALUES ('C016', '顧客016', '台北', '銀卡');
INSERT INTO customers VALUES ('C017', '顧客017', '新竹', '金卡');
INSERT INTO customers VALUES ('C018', '顧客018', '台中', '一般');
INSERT INTO customers VALUES ('C019', '顧客019', '高雄', '銀卡');
INSERT INTO customers VALUES ('C020', '顧客020', '桃園', '金卡');
INSERT INTO customers VALUES ('C021', '顧客021', '台北', '一般');
INSERT INTO customers VALUES ('C022', '顧客022', '新竹', '銀卡');
INSERT INTO customers VALUES ('C023', '顧客023', '台中', '金卡');
INSERT INTO customers VALUES ('C024', '顧客024', '高雄', '一般');
INSERT INTO customers VALUES ('C025', '顧客025', '桃園', '銀卡');

-- 5. orders
CREATE TABLE orders (order_id VARCHAR(10) PRIMARY KEY, customer_id VARCHAR(10) NOT NULL, order_date DATE NOT NULL, status VARCHAR(20) NOT NULL, FOREIGN KEY (customer_id) REFERENCES customers(customer_id));
INSERT INTO orders VALUES ('O0001', 'C004', '2025-02-02', '已完成');
INSERT INTO orders VALUES ('O0002', 'C007', '2025-03-03', '已完成');
INSERT INTO orders VALUES ('O0003', 'C010', '2025-04-04', '已完成');
INSERT INTO orders VALUES ('O0004', 'C013', '2025-05-05', '已完成');
INSERT INTO orders VALUES ('O0005', 'C016', '2025-06-06', '已完成');
INSERT INTO orders VALUES ('O0006', 'C019', '2025-07-07', '已完成');
INSERT INTO orders VALUES ('O0007', 'C022', '2025-08-08', '已取消');
INSERT INTO orders VALUES ('O0008', 'C025', '2025-09-09', '已完成');
INSERT INTO orders VALUES ('O0009', 'C003', '2025-10-10', '已完成');
INSERT INTO orders VALUES ('O0010', 'C006', '2025-11-11', '已完成');
INSERT INTO orders VALUES ('O0011', 'C009', '2025-12-12', '已完成');
INSERT INTO orders VALUES ('O0012', 'C012', '2025-01-13', '已完成');
INSERT INTO orders VALUES ('O0013', 'C015', '2025-02-14', '已完成');
INSERT INTO orders VALUES ('O0014', 'C018', '2025-03-15', '已取消');
INSERT INTO orders VALUES ('O0015', 'C021', '2025-04-16', '已完成');
INSERT INTO orders VALUES ('O0016', 'C024', '2025-05-17', '已完成');
INSERT INTO orders VALUES ('O0017', 'C002', '2025-06-18', '已完成');
INSERT INTO orders VALUES ('O0018', 'C005', '2025-07-19', '已完成');
INSERT INTO orders VALUES ('O0019', 'C008', '2026-08-20', '已完成');
INSERT INTO orders VALUES ('O0020', 'C011', '2026-09-21', '已完成');
INSERT INTO orders VALUES ('O0021', 'C014', '2026-10-22', '已取消');
INSERT INTO orders VALUES ('O0022', 'C017', '2026-11-23', '已完成');
INSERT INTO orders VALUES ('O0023', 'C020', '2026-12-24', '已完成');
INSERT INTO orders VALUES ('O0024', 'C023', '2026-01-25', '已完成');
INSERT INTO orders VALUES ('O0025', 'C001', '2026-02-26', '已完成');
INSERT INTO orders VALUES ('O0026', 'C004', '2026-03-27', '已完成');
INSERT INTO orders VALUES ('O0027', 'C007', '2026-04-01', '已完成');
INSERT INTO orders VALUES ('O0028', 'C010', '2026-05-02', '已取消');
INSERT INTO orders VALUES ('O0029', 'C013', '2026-06-03', '已完成');
INSERT INTO orders VALUES ('O0030', 'C016', '2026-07-04', '已完成');
INSERT INTO orders VALUES ('O0031', 'C019', '2026-08-05', '已完成');
INSERT INTO orders VALUES ('O0032', 'C022', '2026-09-06', '已完成');
INSERT INTO orders VALUES ('O0033', 'C025', '2026-10-07', '已完成');
INSERT INTO orders VALUES ('O0034', 'C003', '2026-11-08', '已完成');
INSERT INTO orders VALUES ('O0035', 'C006', '2026-12-09', '已取消');
INSERT INTO orders VALUES ('O0036', 'C009', '2026-01-10', '已完成');
INSERT INTO orders VALUES ('O0037', 'C012', '2026-02-11', '已完成');
INSERT INTO orders VALUES ('O0038', 'C015', '2026-03-12', '已完成');
INSERT INTO orders VALUES ('O0039', 'C018', '2026-04-13', '已完成');
INSERT INTO orders VALUES ('O0040', 'C021', '2026-05-14', '已完成');
INSERT INTO orders VALUES ('O0041', 'C024', '2026-06-15', '已完成');
INSERT INTO orders VALUES ('O0042', 'C002', '2026-07-16', '已取消');
INSERT INTO orders VALUES ('O0043', 'C005', '2026-08-17', '已完成');
INSERT INTO orders VALUES ('O0044', 'C008', '2026-09-18', '已完成');
INSERT INTO orders VALUES ('O0045', 'C011', '2026-10-19', '已完成');

-- 6. order_items
CREATE TABLE order_items (item_id VARCHAR(10) PRIMARY KEY, order_id VARCHAR(10) NOT NULL, product_id VARCHAR(10) NOT NULL, quantity INT NOT NULL, unit_price DECIMAL(10,2) NOT NULL, FOREIGN KEY (order_id) REFERENCES orders(order_id), FOREIGN KEY (product_id) REFERENCES clothing(product_id));
INSERT INTO order_items VALUES ('D0001', 'O0001', 'P0008', 2, 2150);
INSERT INTO order_items VALUES ('D0002', 'O0001', 'P0019', 3, 390);
INSERT INTO order_items VALUES ('D0003', 'O0002', 'P0015', 3, 1600);
INSERT INTO order_items VALUES ('D0004', 'O0002', 'P0001', 4, 610);
INSERT INTO order_items VALUES ('D0005', 'O0002', 'P0012', 1, 940);
INSERT INTO order_items VALUES ('D0006', 'O0003', 'P0022', 4, 1050);
INSERT INTO order_items VALUES ('D0007', 'O0004', 'P0004', 1, 1270);
INSERT INTO order_items VALUES ('D0008', 'O0004', 'P0015', 2, 1600);
INSERT INTO order_items VALUES ('D0009', 'O0005', 'P0011', 2, 720);
INSERT INTO order_items VALUES ('D0010', 'O0005', 'P0022', 3, 1050);
INSERT INTO order_items VALUES ('D0011', 'O0005', 'P0008', 4, 2150);
INSERT INTO order_items VALUES ('D0012', 'O0006', 'P0018', 3, 2260);
INSERT INTO order_items VALUES ('D0013', 'O0007', 'P0025', 4, 1710);
INSERT INTO order_items VALUES ('D0014', 'O0007', 'P0011', 1, 720);
INSERT INTO order_items VALUES ('D0015', 'O0008', 'P0007', 1, 1930);
INSERT INTO order_items VALUES ('D0016', 'O0008', 'P0018', 2, 2260);
INSERT INTO order_items VALUES ('D0017', 'O0008', 'P0004', 3, 1270);
INSERT INTO order_items VALUES ('D0018', 'O0009', 'P0014', 2, 1380);
INSERT INTO order_items VALUES ('D0019', 'O0010', 'P0021', 3, 830);
INSERT INTO order_items VALUES ('D0020', 'O0010', 'P0007', 4, 1930);
INSERT INTO order_items VALUES ('D0021', 'O0011', 'P0003', 4, 1050);
INSERT INTO order_items VALUES ('D0022', 'O0011', 'P0014', 1, 1380);
INSERT INTO order_items VALUES ('D0023', 'O0011', 'P0025', 2, 1710);
INSERT INTO order_items VALUES ('D0024', 'O0012', 'P0010', 1, 500);
INSERT INTO order_items VALUES ('D0025', 'O0013', 'P0017', 2, 2040);
INSERT INTO order_items VALUES ('D0026', 'O0013', 'P0003', 3, 1050);
INSERT INTO order_items VALUES ('D0027', 'O0014', 'P0024', 3, 1490);
INSERT INTO order_items VALUES ('D0028', 'O0014', 'P0010', 4, 500);
INSERT INTO order_items VALUES ('D0029', 'O0014', 'P0021', 1, 830);
INSERT INTO order_items VALUES ('D0030', 'O0015', 'P0006', 4, 1710);
INSERT INTO order_items VALUES ('D0031', 'O0016', 'P0013', 1, 1160);
INSERT INTO order_items VALUES ('D0032', 'O0016', 'P0024', 2, 1490);
INSERT INTO order_items VALUES ('D0033', 'O0017', 'P0020', 2, 610);
INSERT INTO order_items VALUES ('D0034', 'O0017', 'P0006', 3, 1710);
INSERT INTO order_items VALUES ('D0035', 'O0017', 'P0017', 4, 2040);
INSERT INTO order_items VALUES ('D0036', 'O0018', 'P0002', 3, 830);
INSERT INTO order_items VALUES ('D0037', 'O0019', 'P0023', 4, 1270);
INSERT INTO order_items VALUES ('D0038', 'O0019', 'P0036', 1, 2040);
INSERT INTO order_items VALUES ('D0039', 'O0020', 'P0028', 1, 2370);
INSERT INTO order_items VALUES ('D0040', 'O0020', 'P0041', 2, 1050);
INSERT INTO order_items VALUES ('D0041', 'O0020', 'P0013', 3, 1160);
INSERT INTO order_items VALUES ('D0042', 'O0021', 'P0033', 2, 1380);
INSERT INTO order_items VALUES ('D0043', 'O0022', 'P0038', 3, 390);
INSERT INTO order_items VALUES ('D0044', 'O0022', 'P0010', 4, 500);
INSERT INTO order_items VALUES ('D0045', 'O0023', 'P0043', 4, 1490);
INSERT INTO order_items VALUES ('D0046', 'O0023', 'P0015', 1, 1600);
INSERT INTO order_items VALUES ('D0047', 'O0023', 'P0028', 2, 2370);
INSERT INTO order_items VALUES ('D0048', 'O0024', 'P0048', 1, 500);
INSERT INTO order_items VALUES ('D0049', 'O0025', 'P0012', 2, 940);
INSERT INTO order_items VALUES ('D0050', 'O0025', 'P0025', 3, 1710);
INSERT INTO order_items VALUES ('D0051', 'O0026', 'P0017', 3, 2040);
INSERT INTO order_items VALUES ('D0052', 'O0026', 'P0030', 4, 720);
INSERT INTO order_items VALUES ('D0053', 'O0026', 'P0043', 1, 1490);
INSERT INTO order_items VALUES ('D0054', 'O0027', 'P0022', 4, 1050);
INSERT INTO order_items VALUES ('D0055', 'O0028', 'P0027', 1, 2150);
INSERT INTO order_items VALUES ('D0056', 'O0028', 'P0040', 2, 830);
INSERT INTO order_items VALUES ('D0057', 'O0029', 'P0032', 2, 1160);
INSERT INTO order_items VALUES ('D0058', 'O0029', 'P0045', 3, 1930);
INSERT INTO order_items VALUES ('D0059', 'O0029', 'P0017', 4, 2040);
INSERT INTO order_items VALUES ('D0060', 'O0030', 'P0037', 3, 2260);
INSERT INTO order_items VALUES ('D0061', 'O0031', 'P0042', 4, 1270);
INSERT INTO order_items VALUES ('D0062', 'O0031', 'P0014', 1, 1380);
INSERT INTO order_items VALUES ('D0063', 'O0032', 'P0047', 1, 2370);
INSERT INTO order_items VALUES ('D0064', 'O0032', 'P0019', 2, 390);
INSERT INTO order_items VALUES ('D0065', 'O0032', 'P0032', 3, 1160);
INSERT INTO order_items VALUES ('D0066', 'O0033', 'P0011', 2, 720);
INSERT INTO order_items VALUES ('D0067', 'O0034', 'P0016', 3, 1820);
INSERT INTO order_items VALUES ('D0068', 'O0034', 'P0029', 4, 500);
INSERT INTO order_items VALUES ('D0069', 'O0035', 'P0021', 4, 830);
INSERT INTO order_items VALUES ('D0070', 'O0035', 'P0034', 1, 1600);
INSERT INTO order_items VALUES ('D0071', 'O0035', 'P0047', 2, 2370);
INSERT INTO order_items VALUES ('D0072', 'O0036', 'P0026', 1, 1930);
INSERT INTO order_items VALUES ('D0073', 'O0037', 'P0031', 2, 940);
INSERT INTO order_items VALUES ('D0074', 'O0037', 'P0044', 3, 1710);
INSERT INTO order_items VALUES ('D0075', 'O0038', 'P0036', 3, 2040);
INSERT INTO order_items VALUES ('D0076', 'O0038', 'P0049', 4, 720);
INSERT INTO order_items VALUES ('D0077', 'O0038', 'P0021', 1, 830);
INSERT INTO order_items VALUES ('D0078', 'O0039', 'P0041', 4, 1050);
INSERT INTO order_items VALUES ('D0079', 'O0040', 'P0046', 1, 2150);
INSERT INTO order_items VALUES ('D0080', 'O0040', 'P0018', 2, 2260);
INSERT INTO order_items VALUES ('D0081', 'O0041', 'P0010', 2, 500);
INSERT INTO order_items VALUES ('D0082', 'O0041', 'P0023', 3, 1270);
INSERT INTO order_items VALUES ('D0083', 'O0041', 'P0036', 4, 2040);
INSERT INTO order_items VALUES ('D0084', 'O0042', 'P0015', 3, 1600);
INSERT INTO order_items VALUES ('D0085', 'O0043', 'P0020', 4, 610);
INSERT INTO order_items VALUES ('D0086', 'O0043', 'P0033', 1, 1380);
INSERT INTO order_items VALUES ('D0087', 'O0044', 'P0025', 1, 1710);
INSERT INTO order_items VALUES ('D0088', 'O0044', 'P0038', 2, 390);
INSERT INTO order_items VALUES ('D0089', 'O0044', 'P0010', 3, 500);
INSERT INTO order_items VALUES ('D0090', 'O0045', 'P0030', 2, 720);"""

# 優化 SQL 語句拆分邏輯：過濾掉註解，並移除空語句
raw_statements = raw_sql.split(";")
statements = []
for s in raw_statements:
    lines = [line.strip() for line in s.splitlines()]
    # 過濾掉開頭是 -- 的註解行
    clean_lines = [line for line in lines if line and not line.startswith("--")]
    clean_stmt = " ".join(clean_lines).strip()
    if clean_stmt:
        statements.append(clean_stmt)

with engine.begin() as conn:
    for stmt in statements:
        conn.execute(text(stmt))

print("全部資料表與資料寫入 Filess.io 成功！")
with engine.connect() as conn:
    for tbl in ['categories', 'brands', 'clothing', 'customers', 'orders', 'order_items']:
        cnt = conn.execute(text(f"SELECT COUNT(*) FROM {tbl}")).scalar()
        print(f"{tbl:12s}: {cnt} 筆")

全部資料表與資料寫入 Filess.io 成功！
categories  : 6 筆
brands      : 5 筆
clothing    : 60 筆
customers   : 25 筆
orders      : 45 筆
order_items : 90 筆


In [11]:
# 步驟 2：執行作業題目 A1 ~ B3 查詢測試
sql_tests = {
    "A1_上衣依價格降冪": "SELECT c.product_name, c.price FROM clothing c JOIN categories cat ON c.category_id = cat.category_id WHERE cat.category_name = '上衣' ORDER BY c.price DESC;",
    "A2_UrbanLeaf超過1000元": "SELECT c.product_id, c.product_name, c.price FROM clothing c JOIN brands b ON c.brand_id = b.brand_id WHERE b.brand_name = 'Urban Leaf' AND c.price > 1000;",
    "A3_已完成訂單不重複顧客": "SELECT DISTINCT cu.customer_name FROM customers cu JOIN orders o ON cu.customer_id = o.customer_id WHERE o.status = '已完成' ORDER BY cu.customer_name;",
    "A4_2025有售2026未售商品": "SELECT DISTINCT oi.product_id FROM order_items oi JOIN orders o ON oi.order_id = o.order_id WHERE o.status = '已完成' AND YEAR(o.order_date) = 2025 AND NOT EXISTS (SELECT 1 FROM order_items oi2 JOIN orders o2 ON oi2.order_id = o2.order_id WHERE o2.status = '已完成' AND YEAR(o2.order_date) = 2026 AND oi2.product_id = oi.product_id) ORDER BY oi.product_id;",
    "A5_名稱含外套": "SELECT c.product_name, b.brand_name, c.price FROM clothing c JOIN brands b ON c.brand_id = b.brand_id WHERE c.product_name LIKE '%外套%';",
    "B1_商品超過2款之分類": "SELECT cat.category_name, COUNT(c.product_id) AS product_count, ROUND(AVG(c.price), 2) AS avg_price FROM categories cat JOIN clothing c ON cat.category_id = c.category_id GROUP BY cat.category_id, cat.category_name HAVING COUNT(c.product_id) > 2;",
    "B2_定價最高商品": "SELECT product_name, price FROM clothing WHERE price = (SELECT MAX(price) FROM clothing);",
    "B3_所有商品累計銷售件數": "SELECT c.product_id, c.product_name, COALESCE(SUM(CASE WHEN o.status = '已完成' THEN oi.quantity ELSE 0 END), 0) AS total_sold FROM clothing c LEFT JOIN order_items oi ON c.product_id = oi.product_id LEFT JOIN orders o ON oi.order_id = o.order_id GROUP BY c.product_id, c.product_name ORDER BY c.product_id;"
}

for title, q in sql_tests.items():
    print(f"\n=== {title} ===")
    display(pd.read_sql(q, engine).head(10))


=== A1_上衣依價格降冪 ===


,product_name,price
0,襯衫 37款,2260.0
1,條紋上衣 55款,2040.0
2,襯衫 07款,1930.0
3,條紋上衣 25款,1710.0
4,針織上衣 43款,1490.0
5,針織上衣 13款,1160.0
6,純棉T恤 31款,940.0
7,短袖上衣 49款,720.0
8,純棉T恤 01款,610.0
9,短袖上衣 19款,390.0



=== A2_UrbanLeaf超過1000元 ===


,product_id,product_name,price
0,P0006,運動上衣 06款,1710.0
1,P0016,連帽外套 16款,1820.0
2,P0026,棉質短褲 26款,1930.0
3,P0036,運動上衣 36款,2040.0
4,P0041,無袖洋裝 41款,1050.0
5,P0046,連帽外套 46款,2150.0
6,P0051,A字裙 51款,1160.0
7,P0056,棉質短褲 56款,2260.0



=== A3_已完成訂單不重複顧客 ===


,customer_name
0,顧客001
1,顧客002
2,顧客003
3,顧客004
4,顧客005
5,顧客006
6,顧客007
7,顧客008
8,顧客009
9,顧客010



=== A4_2025有售2026未售商品 ===


,product_id
0,P0001
1,P0002
2,P0003
3,P0004
4,P0006
5,P0007
6,P0008
7,P0024



=== A5_名稱含外套 ===


ValueError: unsupported format character '?' (0x5916) at index 130

In [10]:
# 步驟 3：執行 C1 預留資料表建立、兩筆合法新增與三種失敗案例測試
with engine.begin() as conn:
    conn.execute(text("DROP TABLE IF EXISTS stock_reservation;"))
    conn.execute(text("""
        CREATE TABLE stock_reservation (
            reservation_id INT AUTO_INCREMENT PRIMARY KEY,
            customer_id VARCHAR(10) NOT NULL,
            product_id VARCHAR(10) NOT NULL,
            reserve_date DATE NOT NULL,
            quantity INT NOT NULL CHECK (quantity BETWEEN 1 AND 4),
            CONSTRAINT fk_res_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id) ON DELETE CASCADE,
            CONSTRAINT fk_res_product FOREIGN KEY (product_id) REFERENCES clothing(product_id) ON DELETE CASCADE,
            CONSTRAINT uq_customer_product_date UNIQUE (customer_id, product_id, reserve_date)
        );
    """))
    conn.execute(text("INSERT INTO stock_reservation (customer_id, product_id, reserve_date, quantity) VALUES ('C001', 'P0001', '2026-11-25', 2);"))
    conn.execute(text("INSERT INTO stock_reservation (customer_id, product_id, reserve_date, quantity) VALUES ('C001', 'P0002', '2026-11-25', 3);"))

print("=== C1 兩筆合法預留寫入成功 ===")
display(pd.read_sql("SELECT * FROM stock_reservation;", engine))

fail_cases = [
    ("失敗案例 1：重複預留（違反 UNIQUE 約束）", "INSERT INTO stock_reservation (customer_id, product_id, reserve_date, quantity) VALUES ('C001', 'P0001', '2026-11-25', 1);"),
    ("失敗案例 2：數量為 6（違反 CHECK 1~4 約束）", "INSERT INTO stock_reservation (customer_id, product_id, reserve_date, quantity) VALUES ('C002', 'P0001', '2026-11-25', 6);"),
    ("失敗案例 3：商品不存在 P9999（違反外鍵 FK 約束）", "INSERT INTO stock_reservation (customer_id, product_id, reserve_date, quantity) VALUES ('C001', 'P9999', '2026-11-25', 1);")
]

for label, sql_stmt in fail_cases:
    print(f"\n--- {label} ---")
    try:
        with engine.begin() as conn:
            conn.execute(text(sql_stmt))
    except SQLAlchemyError as e:
        print("預期錯誤訊息:", e.orig)

=== C1 兩筆合法預留寫入成功 ===


,reservation_id,customer_id,product_id,reserve_date,quantity
0,1,C001,P0001,2026-11-25,2
1,2,C001,P0002,2026-11-25,3



--- 失敗案例 1：重複預留（違反 UNIQUE 約束） ---
預期錯誤訊息: (1062, "Duplicate entry 'C001-P0001-2026-11-25' for key 'stock_reservation.uq_customer_product_date'")

--- 失敗案例 2：數量為 6（違反 CHECK 1~4 約束） ---
預期錯誤訊息: (3819, "Check constraint 'stock_reservation_chk_1' is violated.")

--- 失敗案例 3：商品不存在 P9999（違反外鍵 FK 約束） ---
預期錯誤訊息: (1452, 'Cannot add or update a child row: a foreign key constraint fails (`uuuuuuuu_boxrealize`.`stock_reservation`, CONSTRAINT `fk_res_product` FOREIGN KEY (`product_id`) REFERENCES `clothing` (`product_id`) ON DELETE CASCADE)')


In [12]:
# 重新驗證並查詢步驟 3 建立的預留資料表狀態
print("=== 驗證儲存格 5VpmCRF9fGJp 執行結果 ===")
with engine.connect() as conn:
    df_res = pd.read_sql("SELECT * FROM stock_reservation;", conn)
    display(df_res)


=== 驗證儲存格 5VpmCRF9fGJp 執行結果 ===


,reservation_id,customer_id,product_id,reserve_date,quantity
0,1,C001,P0001,2026-11-25,2
1,2,C001,P0002,2026-11-25,3


In [13]:
# 步驟 4：執行 C2 品牌總銷售額超過 10,000 元查詢（含 FUNCTION 測試與等價查詢）
try:
    with engine.begin() as conn:
        conn.execute(text("DROP FUNCTION IF EXISTS total_sales;"))
        conn.execute(text("""
            CREATE FUNCTION total_sales(p_brand_id INT)
            RETURNS DECIMAL(12,2)
            DETERMINISTIC
            READS SQL DATA
            BEGIN
                DECLARE v_total DECIMAL(12,2);
                SELECT COALESCE(SUM(oi.quantity * oi.unit_price), 0)
                INTO v_total
                FROM clothing c
                JOIN order_items oi ON c.product_id = oi.product_id
                JOIN orders o ON oi.order_id = o.order_id
                WHERE c.brand_id = p_brand_id AND o.status = '已完成';
                RETURN v_total;
            END;
        """))
    print("=== C2 使用 total_sales() 函數查詢結果 ===")
    display(pd.read_sql("SELECT brand_id, brand_name, total_sales(brand_id) AS total_revenue FROM brands WHERE total_sales(brand_id) > 10000;", engine))
except SQLAlchemyError as e:
    print("免費版主機權限提示:", e.orig)
    print("改執行 C2 等價 JOIN 查詢:")
    display(pd.read_sql("""
        SELECT b.brand_id, b.brand_name, SUM(oi.quantity * oi.unit_price) AS total_revenue
        FROM brands b
        JOIN clothing c ON b.brand_id = c.brand_id
        JOIN order_items oi ON c.product_id = oi.product_id
        JOIN orders o ON oi.order_id = o.order_id
        WHERE o.status = '已完成'
        GROUP BY b.brand_id, b.brand_name
        HAVING SUM(oi.quantity * oi.unit_price) > 10000;
    """, engine))

免費版主機權限提示: (1419, 'You do not have the SUPER privilege and binary logging is enabled (you *might* want to use the less safe log_bin_trust_function_creators variable)')
改執行 C2 等價 JOIN 查詢:


,brand_id,brand_name,total_revenue
0,3,Mode Studio,67990.0
1,4,Daily Wear,25540.0
2,5,Peak Active,38630.0
3,1,Urban Leaf,54650.0
4,2,North Thread,73060.0


In [14]:
# 步驟 5：匯出前端 Dashboard API 所需的 products.json
query_api = """
SELECT c.product_id, c.product_name, cat.category_name, b.brand_name, b.country AS brand_country,
       c.color, c.size, c.price, c.stock_qty, c.listed_date, c.image_url,
       COALESCE(SUM(CASE WHEN o.status = '已完成' THEN oi.quantity ELSE 0 END), 0) AS sold_qty
FROM clothing c
JOIN categories cat ON c.category_id = cat.category_id
JOIN brands b ON c.brand_id = b.brand_id
LEFT JOIN order_items oi ON c.product_id = oi.product_id
LEFT JOIN orders o ON oi.order_id = o.order_id
GROUP BY c.product_id, c.product_name, cat.category_name, b.brand_name, b.country,
         c.color, c.size, c.price, c.stock_qty, c.listed_date, c.image_url
ORDER BY c.product_id;
"""
df_api = pd.read_sql(query_api, engine)
df_api['listed_date'] = df_api['listed_date'].astype(str)
df_api['price'] = df_api['price'].astype(float)
df_api['sold_qty'] = df_api['sold_qty'].astype(int)
df_api.to_json("products.json", orient="records", force_ascii=False, indent=2)
print(f"已成功匯出 {len(df_api)} 筆商品至 products.json")

已成功匯出 60 筆商品至 products.json
